In [20]:
import os
import xarray as xr
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import logger
import xesmf as xe

import climtas.nci
from dask.distributed import Client, as_completed

In [21]:
LOG = logger.get_logger(__name__)

In [22]:
def himawari_era5_diff(date):

    date_dt = datetime.strptime(date, "%m-%Y")
    year = date_dt.strftime("%Y")
    month = date_dt.strftime("%m")
    
    ##### Himawari Data
    if date_dt <= datetime.strptime('2019-03-31', '%Y-%m-%d'):
        version = 'v1.0'
    else:
        version = 'v1.1'
    directory=Path(f'/g/data/rv74/satellite-products/arc/der/himawari-ahi/solar/p1d/{version}/{year}/{month}')
    files = sorted(str(p) for p in directory.rglob("*.nc"))
    def _preprocess(ds):
        return ds.drop_vars(set(ds.data_vars) - {'daily_integral_of_surface_global_irradiance'})
    
    himawari = xr.open_mfdataset(
        files,
        combine='by_coords',
        preprocess=_preprocess,
        # engine='netcdf4',
    )
    LOG.info('OPEN HIMAWARI')
    himawari = himawari.chunk({'time':10})

    ##### ERA5 Data
    era5_dir = [
        Path(f"/g/data/rt52/era5/single-levels/reanalysis/msdwswrf/{year}"),
    ]
    era5_file = [f for d in era5_dir for f in d.glob(f"msdwswrf_era5_oper_sfc_{year}{month}*.nc")][0]
    era5 = xr.open_dataset(era5_file, chunks={'time':100})
    # era5 = era5.isel(time=slice((int(day)-1)*24,int(day)*24))
    LOG.info('OPEN ERA5')
    
    # Restrict ERA5 domain to Himawari domain
    era5_aus = era5.sel(
        latitude=slice(-10, -44.5),
        longitude=slice(112, 156.26)
    )
    LOG.info(f'ERA5 SLICED SHAPE: {era5_aus.msdwswrf.shape}')

    # Regrid Himawari to more coarse ERA5
    himawari = himawari.interp(
        latitude=era5.latitude,
        longitude=era5.longitude,
        method='linear'
    )
    
    # Get monthly mean
    himawari_clim = himawari.daily_integral_of_surface_global_irradiance.mean(dim='time')
    era5_clim = era5_aus.msdwswrf.mean(dim='time')
    LOG.info(f'HIMARI_CLIM SHAPE: {himawari_clim.shape}')
    LOG.info(f'ERA5_CLIM SHAPE: {era5_clim.shape}')

    
    diff = era5_clim - himawari_clim

    file_name = f'msdwswrf-era5-himawari_{year}{month}.nc'
    file_path = Path('/g/data/er8/users/cd3022/Irradiance-comparisons/era5-himawari/monthly/')
    file_path.mkdir(parents=True, exist_ok=True)
    diff.to_netcdf(file_path / file_name)
    return f"Finished processing {year}-{month}"

In [23]:
client = Client(
    n_workers=10,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 10
Total threads: 10,Total memory: 95.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35649,Workers: 10
Dashboard: /proxy/8787/status,Total threads: 10
Started: Just now,Total memory: 95.00 GiB
Comm: tcp://127.0.0.1:36595,Total threads: 1
Dashboard: /proxy/41615/status,Memory: 9.50 GiB
Nanny: tcp://127.0.0.1:41535,


In [24]:
if __name__ == '__main__':
    futures = {}
    for month in range(1,13):
        date = f'{month}-2016'
        future = client.submit(himawari_era5_diff, date)
        futures[future] = f"The job for {date}"
    for future in as_completed(futures):
        _ = future.result()
        del futures[future]

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /g/data/hh5/public/apps/miniconda3/envs/analysis3-24.04/share/proj failed
ERROR 1: